# 01 — Kiểm kê SPRSound BioCAS2022

**Câu hỏi:** dữ liệu có nguồn rõ ràng, nhãn/thời gian đọc đúng và partition theo người phù hợp với bài Normal/Wheeze không?

Đọc Markdown → chạy ô code bằng Shift+Enter → xem kết quả. Trước khi xác nhận, **Restart Kernel and Run All Cells**. Notebook độc lập với biến của notebook 00.

**Đầu vào:** BioCAS2022 gốc + README chính thức đã tải + báo cáo checksum ngày tải. **Đầu ra:** bảng ngay dưới ô và một bộ CSV/JSON/Markdown trong thư mục audit mới mỗi lần chạy.

**Phạm vi:** đọc annotation/ID/nhãn để audit task và partition định trước. Chỉ đọc WAV header của official training để kiểm tra thời gian; không đọc/giải mã test audio, không tính descriptor hay kết quả classifier test. Intra-subject chỉ kiểm tra provenance và được đánh dấu unused trong core.

Các event `included=true` mới đủ **annotation eligibility**, chưa phải segment đã crop hoặc anchor đã qua ridge review. Chưa chia train/validation và chưa freeze split.

Code dùng thư viện chuẩn Python và IPython.display có cùng Jupyter; không cần cài pandas hay thư viện DSP để chạy notebook này.


## 1. Kernel và thư mục dự án

Để PROJECT_PATH trống khi kernel nằm trong dự án; nếu cần, điền thư mục gốc như ở notebook 00. Các dòng đầu giúp nhận ra kernel Windows/Linux khác nhau.

In [1]:
import csv
import hashlib
import json
import math
import platform
from collections import Counter, defaultdict
from datetime import datetime, timezone
from html import escape
from itertools import combinations
from pathlib import Path, PureWindowsPath
import sys
import uuid
import wave
from IPython.display import display, HTML

PROJECT_PATH = ""  # Ví dụ Windows: r"C:\path\to\DSP-GAP"
current_dir = Path.cwd().resolve()
print("Thư mục kernel:", current_dir)
print("Hệ điều hành:", sys.platform)
print("Python:", sys.executable)
if PROJECT_PATH and sys.platform != "win32" and PureWindowsPath(PROJECT_PATH).drive:
    raise ValueError("Kernel Linux không đọc trực tiếp D:. Chọn Python Windows hoặc đường dẫn mount thật.")
candidates = [Path(PROJECT_PATH).expanduser().resolve()] if PROJECT_PATH else [current_dir, *current_dir.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "requirements-dsp.txt").is_file() and (p / "dsp/notebooks").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Không tìm được project. Điền PROJECT_PATH và kiểm tra kernel như notebook 00.")
DATASET_ROOT = PROJECT_ROOT / "dsp/data/raw/BioCAS2022"
print("Dataset:", DATASET_ROOT)

Thư mục kernel: [project]\dsp\notebooks
Hệ điều hành: win32
Python: [home]\AppData\Local\Programs\Python\Python311\python.exe
Dataset: [project]\dsp\data\raw\BioCAS2022


## 2. Quy tắc audit và nguồn

Subject ID là **patient_number**, phần đầu tên file, theo README chính thức đã tải, không phải suy đoán. `recording_id` dùng toàn bộ filename stem để tránh nhầm recording number cuối tên với ID toàn cục.

Giữ nguyên nhãn nguồn. Chỉ Normal/Wheeze được xét; Poor Quality bị loại theo proposal. Nhãn record (CAS/DAS...) khác nhãn event. Thời gian start/end là chuỗi số theo **ms**. Các ngưỡng DSP chưa được đặt ở đây.

`PARTITIONS` là nhãn nguồn, không phải split mới. Intra-subject giữ trong inventory nhưng mọi event của nó có `included=false` trong core.

In [2]:
PARTITIONS = {
    "official_train": ("train2022_json", "train2022_wav"),
    "inter_test": ("test2022_json/inter_test_json", "test2022_wav"),
    "intra_test_unused": ("test2022_json/intra_test_json", "test2022_wav"),
}
TARGET_LABELS = {"Normal", "Wheeze"}
EVENT_LABELS = {"Normal", "Wheeze", "Rhonchi", "Stridor", "Coarse Crackle", "Fine Crackle", "Wheeze+Crackle"}
RECORD_LABELS = {"Normal", "CAS", "DAS", "CAS & DAS", "Poor Quality"}
SOURCE_COMMIT = "bca1e51422a42a042441010081519610ef3845d0"
SOURCE_URL = f"https://github.com/SJTU-YONGFU-RESEARCH-GRP/SPRSound/blob/{SOURCE_COMMIT}/BioCAS2022/README.md"
readme_path = DATASET_ROOT / "README.md"
source_text = readme_path.read_text(encoding="utf-8-sig")
assert "patient_number_age_gender_location_recording_number" in source_text
assert "milliseconds" in source_text and "Unique identifier for each patient" in source_text
download_path = PROJECT_ROOT / "dsp/outputs/reports/download_biocas2022.json"
# Tương thích manifest cũ; không thay đổi bản audit/split đã khóa.
if not download_path.is_file():
    download_path = PROJECT_ROOT / "dsp/outputs/reports/download_biocas2022_20261026.json"
if not download_path.is_file():
    raise FileNotFoundError("Chạy python dsp/scripts/download_biocas2022.py trước notebook 01.")
download = json.loads(download_path.read_text(encoding="utf-8"))
assert download["commit"] == SOURCE_COMMIT
download_files = {row["repo_path"]: row for row in download["files"]}
print("Nguồn:", SOURCE_URL)
print("Quy tắc tên file và đơn vị thời gian đã có trong README.")

Nguồn: https://github.com/SJTU-YONGFU-RESEARCH-GRP/SPRSound/blob/bca1e51422a42a042441010081519610ef3845d0/BioCAS2022/README.md
Quy tắc tên file và đơn vị thời gian đã có trong README.


## 3. Hai hàm nhỏ để đọc kết quả

`show_table` hiển thị bảng HTML đã escape nội dung. `add_issue` lưu vấn đề thay vì tự sửa dữ liệu. `ERROR` phải giải quyết trước split; `REVIEW` cần xem ngữ cảnh; `INFO` là ghi nhận không chặn core. Không tự động thay đổi nhãn.

In [3]:
def show_table(rows, columns=None, limit=15):
    """Hiển thị vài dòng; dữ liệu đầy đủ vẫn được giữ để xuất CSV."""
    if not rows:
        print("Không có dòng.")
        return
    columns = columns or list(rows[0])
    header = "".join(f"<th>{escape(str(key))}</th>" for key in columns)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(row.get(key, '')))}</td>" for key in columns) + "</tr>" for row in rows[:limit])
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))
    if len(rows) > limit:
        print(f"Hiện {limit}/{len(rows)} dòng; xem CSV để đọc đầy đủ.")

issues = []
def add_issue(severity, code, partition, source, detail):
    """Ghi vấn đề metadata; không ghi đè hoặc chỉnh raw."""
    issues.append(dict(severity=severity, code=code, official_partition=partition, source=source, detail=detail))

## 4. Kiểm tra cặp WAV/JSON trước khi đọc annotation

Đếm theo filename stem. Hai bộ test dùng chung thư mục WAV; vì vậy phải so **hợp** của inter + intra với test WAV. Không ép mỗi subset phải khớp toàn bộ test folder.

Kiểm tra checksum JSON khi đọc ở bước 7. Hash WAV dùng từ báo cáo tải, không đọc lại test audio. Không tự tải hoặc sửa file còn thiếu.

In [4]:
annotation_files = {}
pair_rows = []
test_stems = []
for partition, (json_folder, wav_folder) in PARTITIONS.items():
    json_dir, wav_dir = DATASET_ROOT / json_folder, DATASET_ROOT / wav_folder
    if not json_dir.is_dir() or not wav_dir.is_dir():
        raise FileNotFoundError(f"Thiếu thư mục nguồn: {json_dir} hoặc {wav_dir}")
    files = sorted(json_dir.glob("*.json"))
    if not files:
        raise FileNotFoundError(f"Không có JSON trong {json_dir}")
    annotation_files[partition] = files
    wav_stems = {p.stem for p in wav_dir.glob("*.wav")}
    stems = {p.stem for p in files}
    missing = sorted(stems - wav_stems)
    for stem in missing:
        add_issue("ERROR", "missing_wav", partition, stem, "Có JSON nhưng thiếu WAV tương ứng")
    if partition == "official_train":
        for stem in sorted(wav_stems - stems):
            add_issue("ERROR", "missing_json", partition, stem, "Có train WAV nhưng thiếu JSON")
    else:
        test_stems.extend(stems)
    pair_rows.append(dict(official_partition=partition, json_files=len(files), matching_wav=len(stems & wav_stems), missing_wav=len(missing)))

test_wav_stems = {p.stem for p in (DATASET_ROOT / "test2022_wav").glob("*.wav")}
for stem in sorted(test_wav_stems - set(test_stems)):
    add_issue("ERROR", "missing_json", "test", stem, "Test WAV không thuộc bộ JSON inter/intra")
for stem, count in Counter(test_stems).items():
    if count > 1:
        add_issue("ERROR", "record_in_both_test_subsets", "test", stem, "Cùng recording xuất hiện trong cả inter và intra")
show_table(pair_rows)

official_partition,json_files,matching_wav,missing_wav
official_train,1949,1949,0
inter_test,355,355,0
intra_test_unused,379,379,0


## 5. Đọc subject ID và thời gian một cách tường minh

Tên file phải có đủ 5 phần. Subject và recording number giữ dạng chuỗi; không đổi ID qua số thực. Age chỉ kiểm tra định dạng, không dùng để chia hay tạo mẫu.

Hàm thời gian từ chối NaN, infinity, boolean và dữ liệu không phải số. Đơn vị vẫn là ms; chưa chuyển thành sample cho crop.

In [5]:
def parse_filename(stem):
    """Trả subject ID theo README; raise khi cấu trúc tên không đúng."""
    parts = stem.split("_")
    if len(parts) != 5:
        raise ValueError("Tên phải có 5 phần: patient_age_gender_location_recording")
    subject, age, gender, location, number = parts
    if not subject.isdigit() or not number.isdigit() or gender not in {"0", "1"} or location not in {"p1", "p2", "p3", "p4"}:
        raise ValueError("Patient/recording/gender/location không khớp quy tắc BioCAS2022")
    if not math.isfinite(float(age)) or float(age) < 0:
        raise ValueError("Age không hợp lệ")
    return subject

def parse_time_ms(value):
    """Chuyển chuỗi/số thành ms hữu hạn; không chấp nhận boolean."""
    if isinstance(value, bool) or not isinstance(value, (str, int, float)):
        raise ValueError("Thời gian phải là chuỗi số hoặc số")
    result = float(value)
    if not math.isfinite(result):
        raise ValueError("Thời gian phải hữu hạn")
    return result

## 6. Xem một annotation training có sự kiện

Ví dụ được lấy theo thứ tự tên file, không chọn dựa vào âm thanh hoặc kết quả classifier. Đây là minh họa sự khác nhau giữa nhãn record và danh sách event. Không hiển thị ví dụ test riêng lẻ để phát triển phương pháp.

In [6]:
for path in annotation_files["official_train"]:
    example = json.loads(path.read_text(encoding="utf-8-sig"))
    if example.get("event_annotation"):
        print("Recording:", path.stem)
        print("Subject:", parse_filename(path.stem))
        print("Record annotation:", example.get("record_annotation"))
        show_table(example["event_annotation"], limit=4)
        break

Recording: [recording_id]
Subject: [subject_id]
Record annotation: Normal


start,end,type
1079,4933,Normal


## 7. Parser metadata: một dòng recording, một dòng event

Mỗi event có ID `filename:e0001`, gắn với thứ tự gốc trong JSON (đếm từ 1). Parser giữ cả event bị loại để giải thích exclusion.

- Chỉ đọc WAV header của **official_train** bằng `wave.open`, không đọc sample; lấy format/duration để so end_ms với biên recording.
- Không đọc header test: `bounds_checked=false` ở recording test, cần kiểm tra khi đến bước test được phép.
- `included` phản ánh nhãn, quality, interval và phạm vi core ở thời điểm audit. Crop/context/duplicate review có thể dẫn đến cập nhật eligibility có log ở bước sau.
- Mọi `split` vẫn rỗng. Official train chưa phải final train vì còn phải chia validation theo người.

Ô này đọc vài nghìn file nhỏ nên có thể mất một lúc; dòng tiến độ xuất hiện sau mỗi 500 recording.

In [7]:
recordings, events = [], []
for partition, files in annotation_files.items():
    for path in files:
        stem = path.stem
        json_relative = path.relative_to(DATASET_ROOT).as_posix()
        wav_relative = f"{PARTITIONS[partition][1]}/{stem}.wav"
        wav_path = DATASET_ROOT / wav_relative
        subject = ""
        try:
            subject = parse_filename(stem)
        except (ValueError, OverflowError) as error:
            add_issue("ERROR", "invalid_filename", partition, json_relative, str(error))
        try:
            content = path.read_bytes()
            annotation = json.loads(content.decode("utf-8-sig"))
            if not isinstance(annotation, dict) or not isinstance(annotation.get("event_annotation"), list):
                raise ValueError("JSON phải là object có event_annotation dạng list")
        except (OSError, UnicodeError, ValueError) as error:
            add_issue("ERROR", "invalid_json", partition, json_relative, str(error))
            continue
        annotation_sha = hashlib.sha256(content).hexdigest()
        expected_json = download_files.get(f"BioCAS2022/{json_relative}", {})
        if annotation_sha != expected_json.get("sha256"):
            add_issue("ERROR", "annotation_changed", partition, json_relative, "JSON khác checksum ở bản tải")
        record_label = annotation.get("record_annotation", "")
        if record_label not in RECORD_LABELS:
            add_issue("ERROR", "unknown_record_label", partition, json_relative, repr(record_label))
        audio_source = download_files.get(f"BioCAS2022/{wav_relative}", {})
        if not audio_source:
            add_issue("ERROR", "missing_audio_provenance", partition, wav_relative, "Thiếu checksum bản tải")
        elif wav_path.is_file() and wav_path.stat().st_size != audio_source["bytes"]:
            add_issue("ERROR", "audio_size_changed", partition, wav_relative, "Kích thước WAV khác bản tải")
        rate, channels, width, frames, duration_ms = "", "", "", "", None
        bounds_checked = False
        if partition == "official_train" and wav_path.is_file():
            try:
                with wave.open(str(wav_path), "rb") as audio:
                    rate, channels = audio.getframerate(), audio.getnchannels()
                    width, frames = audio.getsampwidth(), audio.getnframes()
                if rate <= 0 or frames <= 0:
                    raise ValueError("Sample rate/frame count không dương")
                duration_ms = frames * 1000 / rate
                bounds_checked = True
            except (wave.Error, OSError, EOFError, ValueError) as error:
                add_issue("ERROR", "invalid_train_wav_header", partition, wav_relative, str(error))
        recording = dict(recording_id=stem, subject_id=subject, official_partition=partition,
                         audio_path=wav_relative, annotation_path=json_relative, record_label=record_label,
                         annotated_events=len(annotation["event_annotation"]), sample_rate_hz=rate,
                         channels=channels, sample_width_bytes=width, n_frames=frames,
                         duration_ms=duration_ms if duration_ms is not None else "",
                         bounds_checked=bounds_checked, annotation_sha256=annotation_sha,
                         audio_sha256_at_download=audio_source.get("sha256", ""))
        recordings.append(recording)
        for index, item in enumerate(annotation["event_annotation"], 1):
            event_id = f"{stem}:e{index:04d}"
            reasons = []
            source_label = item.get("type", "") if isinstance(item, dict) else ""
            if source_label not in EVENT_LABELS:
                add_issue("ERROR", "unknown_event_label", partition, event_id, repr(source_label))
            start, end = "", ""
            try:
                if not isinstance(item, dict):
                    raise ValueError("Event phải là object")
                start, end = parse_time_ms(item.get("start")), parse_time_ms(item.get("end"))
                if not 0 <= start < end:
                    raise ValueError("Cần 0 <= start_ms < end_ms")
                if bounds_checked and end > duration_ms:
                    raise ValueError(f"end_ms={end} vượt duration_ms={duration_ms}")
            except (ValueError, TypeError, OverflowError) as error:
                reasons.append("invalid_interval")
                add_issue("ERROR", "invalid_interval", partition, event_id, str(error))
            if not subject:
                reasons.append("invalid_subject_id")
            if not wav_path.is_file():
                reasons.append("missing_wav")
            if record_label == "Poor Quality":
                reasons.append("poor_quality_record")
            elif record_label not in RECORD_LABELS:
                reasons.append("unverified_record_quality")
            if source_label not in TARGET_LABELS:
                reasons.append("outside_normal_wheeze_task")
            if partition == "intra_test_unused":
                reasons.append("intra_subject_unused_in_core")
            events.append(dict(event_id=event_id, recording_id=stem, subject_id=subject,
                               audio_path=wav_relative, annotation_path=json_relative,
                               start_ms=start, end_ms=end, source_label=source_label,
                               label=source_label if source_label in TARGET_LABELS else "",
                               official_partition=partition, split="", included=not reasons,
                               exclusion_reason=";".join(reasons)))
        if len(recordings) % 500 == 0:
            print(f"Đã đọc {len(recordings)} recording", flush=True)
print("Tổng recording/event đã đọc:", len(recordings), len(events))

Đã đọc 500 recording
Đã đọc 1000 recording
Đã đọc 1500 recording
Đã đọc 2000 recording
Đã đọc 2500 recording
Tổng recording/event đã đọc: 2683 9089


## 8. Event trùng/overlap trong cùng recording

Event giống interval và nhãn là duplicate cần xem lại. Hai interval giao nhau được báo REVIEW, kể cả cùng nhãn: cần xem annotation/crop rule trước khi dùng. Không tự xóa dòng hoặc mở rộng context để che conflict.

Các event sai interval ở bước trước vẫn còn trong manifest để truy vết, nhưng không dùng cho kiểm tra overlap.

In [8]:
by_record = defaultdict(list)
for event in events:
    if "invalid_interval" not in event["exclusion_reason"]:
        by_record[(event["official_partition"], event["recording_id"])].append(event)
overlap_rows = []
for (partition, recording_id), group in by_record.items():
    seen = set()
    ordered = sorted(group, key=lambda e: (e["start_ms"], e["end_ms"], e["event_id"]))
    for event in ordered:
        key = (event["start_ms"], event["end_ms"], event["source_label"])
        if key in seen:
            add_issue("ERROR", "duplicate_event", partition, event["event_id"], "Trùng interval và nhãn trong recording")
        seen.add(key)
    for left, right in combinations(ordered, 2):
        if max(left["start_ms"], right["start_ms"]) < min(left["end_ms"], right["end_ms"]):
            overlap_rows.append(dict(official_partition=partition, recording_id=recording_id,
                                     event_id_left=left["event_id"], event_id_right=right["event_id"],
                                     label_left=left["source_label"], label_right=right["source_label"]))
            add_issue("REVIEW", "overlapping_events", partition, recording_id, f"{left['event_id']} / {right['event_id']}")
print("Số cặp event overlap:", len(overlap_rows))
show_table(overlap_rows)

Số cặp event overlap: 0
Không có dòng.


## 9. Subject intersections và duplicate recording

Train giao inter-test là ERROR. Train giao intra-test là đặc điểm dự kiến của bộ intra, không được dùng intra như test người mới. Bảng xuất giữ cả ID để review nội bộ; không đưa metadata người lên repo công khai.

Duplicate WAV dựa trên SHA-256 **ở thời điểm tải**, không phải phân tích waveform. Kiểm tra kích thước ở bước 7 không chứng minh byte chưa đổi nếu ai đó sửa file mà giữ nguyên kích thước; cần re-verify checksum nếu raw từng bị sửa. Duplicate giữa train/inter là blocker cần xử lý trước split. Nếu chỉ một recording core trùng với cùng người trong intra không dùng, ghi INFO: không có bản sao thứ hai trong tập core. Duplicate trong core hoặc khác người vẫn cần REVIEW/ERROR.

In [9]:
subject_sets = {part: {r["subject_id"] for r in recordings if r["official_partition"] == part and r["subject_id"]} for part in PARTITIONS}
intersection_rows = []
for left, right in combinations(PARTITIONS, 2):
    shared = sorted(subject_sets[left] & subject_sets[right])
    intersection_rows.append(dict(partition_left=left, partition_right=right, shared_subjects=len(shared), subject_ids=";".join(shared)))
    if shared and "intra_test_unused" not in {left, right}:
        add_issue("ERROR", "subject_leakage", left, right, f"{len(shared)} subject trùng giữa core partitions")
show_table(intersection_rows, columns=["partition_left", "partition_right", "shared_subjects"])

def duplicate_severity(group):
    """Phân biệt rò rỉ core, duplicate cần review, và bản sao trong intra unused."""
    core = [row for row in group if row["official_partition"] != "intra_test_unused"]
    core_parts = {row["official_partition"] for row in core}
    if {"official_train", "inter_test"} <= core_parts:
        return "ERROR"
    if len(core) <= 1 and len({row["subject_id"] for row in group}) == 1:
        return "INFO"
    return "REVIEW"

hash_groups = defaultdict(list)
for recording in recordings:
    if recording["audio_sha256_at_download"]:
        hash_groups[recording["audio_sha256_at_download"]].append(recording)
duplicate_rows = []
for sha, group in hash_groups.items():
    if len(group) < 2:
        continue
    parts = {r["official_partition"] for r in group}
    for recording in group:
        duplicate_rows.append(dict(audio_sha256_at_download=sha, recording_id=recording["recording_id"], subject_id=recording["subject_id"], official_partition=recording["official_partition"]))
    severity = duplicate_severity(group)
    add_issue(severity, "duplicate_wav_at_download", ";".join(sorted(parts)), sha, ";".join(r["recording_id"] for r in group))
print("Recording thuộc nhóm byte-identical ở bản tải:", len(duplicate_rows))
show_table(duplicate_rows, limit=5)

partition_left,partition_right,shared_subjects
official_train,inter_test,0
official_train,intra_test_unused,162
inter_test,intra_test_unused,0


Recording thuộc nhóm byte-identical ở bản tải: 2


audio_sha256_at_download,recording_id,subject_id,official_partition
2ef5c2f392a4cd8c3bb26501cf528ca5107caaba0af146488c800b31edd804a6,[recording_id],[subject_id],official_train
2ef5c2f392a4cd8c3bb26501cf528ca5107caaba0af146488c800b31edd804a6,[recording_id],[subject_id],intra_test_unused


## 10. Bảng số lượng và mất cân bằng lớp

`subjects`/`wheeze_subjects` mới nói về đơn vị người độc lập. Số event không phải số người. `eligible_events` dùng inclusion sơ bộ, chưa phải số segment cuối.

Intra có eligible_events=0 do không dùng trong core; các nhãn gốc vẫn được đếm riêng. Đây là label inventory để xác lập task, không dùng chọn generator/budget hoặc split theo điểm số.

In [10]:
summary_rows, label_rows, subject_rows = [], [], []
for partition in PARTITIONS:
    part_recordings = [r for r in recordings if r["official_partition"] == partition]
    part_events = [e for e in events if e["official_partition"] == partition]
    eligible = [e for e in part_events if e["included"]]
    summary_rows.append(dict(official_partition=partition, subjects=len(subject_sets[partition]),
                             recordings=len(part_recordings), poor_quality_recordings=sum(r["record_label"] == "Poor Quality" for r in part_recordings),
                             annotated_events=len(part_events), eligible_events=len(eligible),
                             normal_events=sum(e["label"] == "Normal" for e in eligible),
                             wheeze_events=sum(e["label"] == "Wheeze" for e in eligible),
                             wheeze_subjects=len({e["subject_id"] for e in eligible if e["label"] == "Wheeze"})))
    for label, count in sorted(Counter(e["source_label"] for e in part_events).items()):
        label_rows.append(dict(official_partition=partition, source_label=label, annotated_events=count))
    for subject in sorted(subject_sets[partition]):
        selected = [e for e in eligible if e["subject_id"] == subject]
        subject_rows.append(dict(subject_id=subject, official_partition=partition,
                                 recordings=sum(r["subject_id"] == subject for r in part_recordings),
                                 normal_events=sum(e["label"] == "Normal" for e in selected),
                                 wheeze_events=sum(e["label"] == "Wheeze" for e in selected), split=""))
show_table(summary_rows)
show_table(label_rows, limit=25)
train_counts = next(row for row in summary_rows if row["official_partition"] == "official_train")
if train_counts["wheeze_events"]:
    print("Official train — tỷ lệ event Normal/Wheeze:", round(train_counts["normal_events"] / train_counts["wheeze_events"], 2))
    print("Chưa dùng tỷ lệ này để chốt B: phải chia final train/validation và audit crop trước.")
print("Định dạng header official train:", dict(Counter((r["sample_rate_hz"], r["channels"], r["sample_width_bytes"]) for r in recordings if r["official_partition"] == "official_train")))

official_partition,subjects,recordings,poor_quality_recordings,annotated_events,eligible_events,normal_events,wheeze_events,wheeze_subjects
official_train,251,1949,177,6656,5611,5159,452,50
inter_test,41,355,8,1429,1345,1040,305,13
intra_test_unused,162,379,2,1004,0,0,0,0


official_partition,source_label,annotated_events
official_train,Coarse Crackle,49
official_train,Fine Crackle,912
official_train,Normal,5159
official_train,Rhonchi,39
official_train,Stridor,15
official_train,Wheeze,452
official_train,Wheeze+Crackle,30
inter_test,Coarse Crackle,3
inter_test,Fine Crackle,80
inter_test,Normal,1040


Official train — tỷ lệ event Normal/Wheeze: 11.41
Chưa dùng tỷ lệ này để chốt B: phải chia final train/validation và audit crop trước.
Định dạng header official train: {(8000, 1, 2): 1949}


## 11. Báo lỗi và tự kiểm tra các ràng buộc

Hàm parser được thử trên ca hợp lệ và ca sai để không chỉ tin vào dữ liệu hiện đang đọc được. Không đặt một số người tối thiểu tùy ý: bước split tiếp theo mới thử đúng quy tắc seeds trong proposal.

`ready_for_split=false` nghĩa là giữ báo cáo, giải quyết ERROR/REVIEW trước khi freeze split. Nó không làm mất các dòng bị loại hoặc tự thay partition.

In [11]:
assert parse_filename("65058931_3.4_1_p2_1086") == "65058931"
assert parse_time_ms("2499") == 2499
for bad in [True, "nan", "inf", None, "not-a-number"]:
    try:
        parse_time_ms(bad)
    except (ValueError, TypeError):
        pass
    else:
        raise AssertionError(f"Parser chấp nhận thời gian sai: {bad!r}")
assert len({e["event_id"] for e in events}) == len(events), "Event IDs bị trùng"
assert all(e["split"] == "" for e in events), "Notebook audit không được tự chia split"
assert all(not e["included"] for e in events if e["official_partition"] == "intra_test_unused")
assert all(e["label"] in TARGET_LABELS for e in events if e["included"])
assert all(not r["bounds_checked"] for r in recordings if r["official_partition"] != "official_train")
# Probe logic duplicate: khác official partition nhưng intra unused không làm rò rỉ core.
train_example = {"official_partition": "official_train", "subject_id": "probe_subject"}
intra_example = {"official_partition": "intra_test_unused", "subject_id": "probe_subject"}
inter_example = {"official_partition": "inter_test", "subject_id": "probe_subject"}
assert duplicate_severity([train_example, intra_example]) == "INFO"
assert duplicate_severity([train_example, train_example]) == "REVIEW"
assert duplicate_severity([train_example, inter_example]) == "ERROR"
info_count = sum(row["severity"] == "INFO" for row in issues)
print("INFO:", info_count)
error_count = sum(row["severity"] == "ERROR" for row in issues)
review_count = sum(row["severity"] == "REVIEW" for row in issues)
ready_for_split = error_count == 0 and review_count == 0
print("ERROR:", error_count, "| REVIEW:", review_count)
print("READY FOR SPLIT REVIEW:", ready_for_split)
show_table(issues, limit=15)

INFO: 1
ERROR: 0 | REVIEW: 0
READY FOR SPLIT REVIEW: True


severity,code,official_partition,source,detail
INFO,duplicate_wav_at_download,intra_test_unused;official_train,2ef5c2f392a4cd8c3bb26501cf528ca5107caaba0af146488c800b31edd804a6,[recording_id];[recording_id]


## 12. Xuất các bảng CSV vào thư mục audit mới

Mỗi lần chạy tạo một `audit_<UTC time>_<random suffix>`. Manifest và báo cáo cùng run_id để notebook sau nối đúng bộ. Bảng subject là **inventory subject × official partition**, chưa phải `subjects.csv` final split (một người có thể có trong train và intra).

Trước khi dùng ở bước tiếp theo, cần đọc `audit_summary.json` và các issues. Cờ ready chỉ áp dụng phạm vi metadata audit; không khẳng định crop/synthesis hoặc split đã hoàn thành. Input checksum, notebook snapshot và môi trường được lưu để truy nguồn.

In [12]:
run_id = "audit_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
manifest_dir = PROJECT_ROOT / "dsp/data/manifests" / run_id
report_dir = PROJECT_ROOT / "dsp/outputs/reports" / run_id
manifest_dir.mkdir(parents=True, exist_ok=False)
report_dir.mkdir(parents=True, exist_ok=False)

def write_csv(path, rows, columns):
    """UTF-8 CSV; ghi header kể cả bảng rỗng, không ghi đè file cũ."""
    with path.open("x", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=columns)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: str(row.get(key, "")).lower() if isinstance(row.get(key), bool) else row.get(key, "") for key in columns})

with (PROJECT_ROOT / "dsp/templates/events.csv").open(encoding="utf-8", newline="") as template:
    event_columns = next(csv.reader(template))
write_csv(manifest_dir / "events.csv", events, event_columns)
write_csv(manifest_dir / "recordings.csv", recordings, list(recordings[0]))
write_csv(manifest_dir / "subject_inventory.csv", subject_rows, list(subject_rows[0]))
write_csv(report_dir / "partition_counts.csv", summary_rows, list(summary_rows[0]))
write_csv(report_dir / "label_counts.csv", label_rows, ["official_partition", "source_label", "annotated_events"])
write_csv(report_dir / "issues.csv", issues, ["severity", "code", "official_partition", "source", "detail"])
write_csv(report_dir / "overlaps.csv", overlap_rows, ["official_partition", "recording_id", "event_id_left", "event_id_right", "label_left", "label_right"])
write_csv(report_dir / "subject_intersections.csv", intersection_rows, ["partition_left", "partition_right", "shared_subjects", "subject_ids"])
write_csv(report_dir / "duplicate_recordings.csv", duplicate_rows, ["audio_sha256_at_download", "recording_id", "subject_id", "official_partition"])

## 13. Lưu nguồn và môi trường để chạy lại

Snapshot notebook bỏ output cũ; manifest đầu vào và output được ghi checksum. Hãy lưu Ctrl+S trước Run All để snapshot trên đĩa đúng với code vừa sửa. Không cần nhớ biến của notebook 00. Phiên bản Python, đường dẫn kernel và quy tắc audit đều nằm trong audit_summary.json.

In [13]:
notebook_path = PROJECT_ROOT / "dsp/notebooks/01_kiem_ke_du_lieu.ipynb"
snapshot = json.loads(notebook_path.read_text(encoding="utf-8"))
for cell in snapshot["cells"]:
    if cell["cell_type"] == "code":
        cell["outputs"], cell["execution_count"] = [], None
with (report_dir / "notebook_source.ipynb").open("x", encoding="utf-8") as file:
    json.dump(snapshot, file, ensure_ascii=False, indent=1)

summary = dict(run_id=run_id, created_at_utc=datetime.now(timezone.utc).isoformat(),
               source_url=SOURCE_URL, source_commit=SOURCE_COMMIT,
               dataset_root=str(DATASET_ROOT), readme_sha256=hashlib.sha256(readme_path.read_bytes()).hexdigest(),
               download_report_sha256=hashlib.sha256(download_path.read_bytes()).hexdigest(),
               notebook_source_sha256=hashlib.sha256((report_dir / "notebook_source.ipynb").read_bytes()).hexdigest(),
               python=sys.version, executable=sys.executable, platform=platform.platform(),
               rules=dict(target_labels=sorted(TARGET_LABELS), exclude_poor_quality=True,
                          intra_subject_core=False, split_assigned=False, test_audio_read=False,
                          eligibility_stage="annotation_only_not_final_segments", waveform_duplicates="hashes_at_download_not_fresh_test_hashes"),
               partition_counts=summary_rows, error_count=error_count, review_count=review_count, info_count=info_count,
               ready_for_split_review=ready_for_split,
               manifest_dir=manifest_dir.relative_to(PROJECT_ROOT).as_posix(),
               limitations=["No train/validation split yet", "No test audio/header inspected", "No crop/context eligibility yet", "No modeling anchor review", "No clinically validated synthesis claims"])
summary["output_sha256"] = {str(path.relative_to(PROJECT_ROOT).as_posix()): hashlib.sha256(path.read_bytes()).hexdigest() for folder in [manifest_dir, report_dir] for path in sorted(folder.iterdir()) if path.is_file()}
with (report_dir / "audit_summary.json").open("x", encoding="utf-8") as file:
    json.dump(summary, file, ensure_ascii=False, indent=2)

## 14. Viết báo cáo ngắn và xem đường dẫn

BAO_CAO.md là bản dễ đọc của cùng run. Sau khi chạy xong, mở báo cáo rồi xem issues.csv nếu có ERROR/REVIEW. Chỉ notebook chia tập tiếp theo mới được gán split; chưa tiến hành crop hay synthesis trong notebook này.

In [14]:
text = ["# Metadata audit BioCAS2022", "", f"Run: {run_id}", f"ERROR: {error_count}; REVIEW: {review_count}; INFO: {info_count}", f"Ready for split review: {ready_for_split}", "", "| Partition | Subjects | Recordings | Events | Eligible | Normal | Wheeze | Wheeze subjects |", "|---|---:|---:|---:|---:|---:|---:|---:|"]
for row in summary_rows:
    text.append("| " + " | ".join(str(row[key]) for key in ["official_partition", "subjects", "recordings", "annotated_events", "eligible_events", "normal_events", "wheeze_events", "wheeze_subjects"]) + " |")
text += ["", "Eligibility is annotation-only; counts are not final segments or independent modeling anchors.", "Test audio not read. No final split, crop, descriptor fitting or classifier evaluation performed.", "See issues.csv, overlaps.csv and audit_summary.json before proceeding."]
with (report_dir / "BAO_CAO.md").open("x", encoding="utf-8") as file:
    file.write(chr(10).join(text) + chr(10))
print("Manifest:", manifest_dir)
print("Báo cáo:", report_dir)
print("Run ID:", run_id)

Manifest: [project]\dsp\data\manifests\audit_20261028T112837Z_bf686e
Báo cáo: [project]\dsp\outputs\reports\audit_20261028T112837Z_bf686e
Run ID: audit_20261028T112837Z_bf686e


## Bạn cần hiểu gì sau khi chạy?

1. Subject, recording, event và segment khác nhau thế nào?
2. Vì sao Normal/Wheeze phải lấy từ event_annotation, và CAS không tự đồng nghĩa Wheeze?
3. Vì sao intra-subject không dùng để tuyên bố khả năng tổng quát trên người mới?
4. Vì sao số wheeze event không thay thế số người có wheeze?
5. Vì sao `included=true` ở đây chưa có nghĩa là được dùng làm modeling anchor?

**Bước tiếp theo:** review issues rồi tạo notebook chia subject train/validation theo seeds 2026–2035 của proposal. Chưa tạo split chỉ vì đã có số lượng tổng. Báo cáo này không dùng nhãn test để tối ưu generator/budget.
